In [1]:
# get base model
# create train process
# create peft model
# check training parameters
# run train process

In [2]:
import sys
sys.path.append("../")

from src.llm.training import TrainingProcess
from src.llm.models import Model
from src.llm.loading import Loader
from pathlib import Path
from src.llm.processing import Processor

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# get base model
model = Model(name="Qwen/Qwen2.5-1.5B-Instruct")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 12364.59it/s]


In [4]:
loader = Loader(path=Path("../data/data.json"), test_k=0.2, seed=42)
ds = loader.load()
train_ds, test_ds = loader.train_test_split(dataset=ds)
train_ds, val_ds = loader.train_test_split(dataset=train_ds)
(len(ds), len(train_ds), len(val_ds), len(test_ds))

(49, 31, 8, 10)

In [5]:
# preprocess datasets
preprocessor = Processor(
    tokenizer=model.tokenizer,
)

preprocess_func = preprocessor.preprocess

train_ds = train_ds.map(preprocess_func, remove_columns=train_ds.column_names)
val_ds = val_ds.map(preprocess_func, remove_columns=val_ds.column_names)

Map: 100%|██████████| 8/8 [00:00<00:00, 1483.66 examples/s]


In [6]:
# create train process
training_process = TrainingProcess(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    output_dir="../model",
    r=8
)

In [7]:
# create peft model
peft_model = training_process._peft_model()
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [8]:
# check training parameters
peft_model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [9]:
# get train args
training_process._training_args()

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [10]:
# get trainer
training_process._trainer()

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [11]:
# run train process
training_process.train_save()

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.018945,0.091718
2,1.129114,1.157934
3,1.181552,0.830217
4,10.243541,6.125653
5,10.069771,10.403290
6,47.131947,37.394703
7,50.086937,40.804596
8,2.606418,2.361145
9,0.866957,0.232162
10,0.002530,0.126772
